In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/imaisalah/test-clean/test_clean.csv
/kaggle/input/datasets/imaisalah/processed2/train_clean.csv
/kaggle/input/datasets/imaisalah/processed2/sentiment_class_weights.json
/kaggle/input/datasets/imaisalah/processed2/train_flat_balanced.csv
/kaggle/input/datasets/imaisalah/processed2/unlabeled_clean.csv
/kaggle/input/datasets/imaisalah/processed2/val_clean.csv
/kaggle/input/datasets/imaisalah/processed2/train_flat.csv
/kaggle/input/datasets/imaisalah/processed2/val_flat.csv


In [16]:
"""
TRACK 2 — MARBERT Fine-tuning + Hybrid Inference
Arabic → MARBERT | Other languages → XLM-RoBERTa
"""

import json
import re
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer
import ast

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
MODEL_NAME      = "UBC-NLP/MARBERT"
XLMR_MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
MAX_LEN         = 128
BATCH_SIZE      = 16
EPOCHS          = 5
LR              = 2e-5

TRAIN_PATH        = "/kaggle/input/datasets/imaisalah/processed2/train_flat_balanced.csv"
TRAIN_CLEAN_PATH  = "/kaggle/input/datasets/imaisalah/processed2/train_clean.csv"   # ← for aspect classifier
VAL_PATH          = "/kaggle/input/datasets/imaisalah/processed2/val_flat.csv"
WEIGHTS_PATH      = "/kaggle/input/datasets/imaisalah/processed2/sentiment_class_weights.json"
TEST_PATH         = "/kaggle/input/datasets/imaisalah/test-clean/test_clean.csv"
OUTPUT_PATH       = "submission.json"

VALID_ASPECTS = ['food', 'service', 'price', 'cleanliness', 'delivery',
                 'ambiance', 'app_experience', 'general', 'none']

# ─────────────────────────────────────────────
# 1. SHARED UTILITIES
# ─────────────────────────────────────────────
def light_clean(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def is_arabic_script(text: str) -> bool:
    """True if >20% of non-space chars are Arabic Unicode."""
    if not isinstance(text, str) or not text.strip():
        return False
    arabic_chars = sum(1 for c in text if "\u0600" <= c <= "\u06FF")
    ratio = arabic_chars / max(len(text.replace(" ", "")), 1)
    return ratio > 0.20

# ── CHANGED: ML-based aspect classifier replaces keyword detection ─────────────

def safe_parse(x):
    try:
        return ast.literal_eval(str(x))
    except Exception:
        return []

def train_aspect_classifier():
    """Train a TF-IDF + LogisticRegression multi-label aspect classifier
    on the review-level labeled training data."""
    train_review = pd.read_csv(TRAIN_CLEAN_PATH)
    train_review['aspects_list'] = train_review['aspects'].apply(safe_parse)
    train_review = train_review[train_review['aspects_list'].apply(len) > 0].copy()
    train_review['text_clean'] = train_review['text_clean'].fillna('').apply(light_clean)

    mlb = MultiLabelBinarizer(classes=VALID_ASPECTS)
    Y   = mlb.fit_transform(train_review['aspects_list'])

    tfidf = TfidfVectorizer(
        max_features=30000,
        ngram_range=(1, 2),
        analyzer='char_wb',
        min_df=2,
    )
    X = tfidf.fit_transform(train_review['text_clean'])

    clf = OneVsRestClassifier(LogisticRegression(C=1, max_iter=1000))
    clf.fit(X, Y)

    print(f"Aspect classifier trained on {len(train_review)} reviews ✓")
    return tfidf, clf, mlb

tfidf, aspect_clf, mlb = train_aspect_classifier()

def detect_aspects(text: str) -> list:
    """Predict aspects using trained ML classifier (replaces keyword matching)."""
    if not isinstance(text, str) or not text.strip():
        return ["general"]
    vec  = tfidf.transform([light_clean(text)])
    pred = aspect_clf.predict(vec)[0]
    aspects = [VALID_ASPECTS[i] for i, v in enumerate(pred) if v == 1]
    # Fallback: if nothing predicted, take the highest-probability aspect
    if not aspects:
        proba   = aspect_clf.predict_proba(vec)[0]
        top_i   = int(proba.argmax())
        aspects = [VALID_ASPECTS[top_i]]
    return aspects

# ─────────────────────────────────────────────
# 2. LOAD TRAINING DATA
# ─────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)

train_df["clean_text"] = train_df["text_clean"].apply(light_clean)
val_df["clean_text"]   = val_df["text_clean"].apply(light_clean)

print(f"Train rows: {len(train_df)} | Val rows: {len(val_df)}")
print("Label distribution (train):\n", train_df["sentiment"].value_counts())

# ─────────────────────────────────────────────
# 3. ENCODE LABELS
# ─────────────────────────────────────────────
LABEL_ORDER = ["negative", "neutral", "positive"]
le = LabelEncoder()
le.fit(LABEL_ORDER)

train_df["label_id"] = le.transform(train_df["sentiment"])
val_df["label_id"]   = le.transform(val_df["sentiment"])
num_labels = len(le.classes_)

# ─────────────────────────────────────────────
# 4. CLASS WEIGHTS
# ─────────────────────────────────────────────
with open(WEIGHTS_PATH) as f:
    raw_weights = json.load(f)

weight_list   = [raw_weights[cls] for cls in le.classes_]
class_weights = torch.tensor(weight_list, dtype=torch.float)
print(f"\nClass weights ({list(le.classes_)}): {weight_list}")

# ─────────────────────────────────────────────
# 5. MARBERT TOKENIZER + DATASETS
# ─────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

def make_dataset(dataframe: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({
        "clean_text": dataframe["clean_text"].tolist(),
        "labels":     dataframe["label_id"].tolist(),
    })
    return ds.map(tokenize, batched=True)

train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df)

# ─────────────────────────────────────────────
# 6. MARBERT MODEL + WEIGHTED TRAINER
# ─────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="single_label_classification",
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits
        loss    = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1":    f1_score(labels, preds, average="macro",    zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }

# ─────────────────────────────────────────────
# 7. TRAIN
# ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./marbert_output",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

# Validation report
raw_preds = trainer.predict(val_ds)
preds     = np.argmax(raw_preds.predictions, axis=1)
print("\nClassification report:")
print(classification_report(val_df["label_id"].values, preds,
                             target_names=le.classes_, zero_division=0))

# ─────────────────────────────────────────────
# 8. LOAD TEST DATA + ROUTE BY LANGUAGE
# ─────────────────────────────────────────────
test_df = pd.read_csv(TEST_PATH)
text_col = "text_clean" if "text_clean" in test_df.columns else "text"
test_df["clean_text"] = test_df[text_col].fillna("").apply(light_clean)   # ← also fixed NaN
test_df["is_arabic"]  = test_df["clean_text"].apply(is_arabic_script)

arabic_df = test_df[test_df["is_arabic"]].copy()
other_df  = test_df[~test_df["is_arabic"]].copy()

print(f"\nTest split → Arabic (MARBERT): {len(arabic_df)} | Other (XLM-R): {len(other_df)}")

# ─────────────────────────────────────────────
# 9. INFERENCE — Arabic via MARBERT
# ─────────────────────────────────────────────
def expand_aspects(df: pd.DataFrame) -> pd.DataFrame:
    # ── CHANGED: uses detect_aspects (ML classifier) instead of keyword matching
    records = []
    for _, row in df.iterrows():
        for aspect in detect_aspects(row["clean_text"]):
            records.append({"review_id": row["review_id"],
                            "aspect": aspect, "clean_text": row["clean_text"]})
    return pd.DataFrame(records) if records else pd.DataFrame(
        columns=["review_id", "aspect", "clean_text"])

def run_marbert(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["review_id", "aspect", "sentiment"])
    infer_df = expand_aspects(df)
    ds = Dataset.from_dict({"clean_text": infer_df["clean_text"].tolist(),
                             "labels": [0] * len(infer_df)})
    ds = ds.map(tokenize, batched=True)
    raw  = trainer.predict(ds)
    infer_df["sentiment"] = le.inverse_transform(np.argmax(raw.predictions, axis=1))
    return infer_df

# ─────────────────────────────────────────────
# 10. INFERENCE — Non-Arabic via XLM-RoBERTa
# ─────────────────────────────────────────────
XLMR_LABEL_MAP = {
    "Negative": "negative", "Neutral": "neutral", "Positive": "positive",
    "LABEL_0":  "negative", "LABEL_1": "neutral", "LABEL_2":  "positive",
}

def run_xlmr(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["review_id", "aspect", "sentiment"])

    xlmr_tok   = AutoTokenizer.from_pretrained(XLMR_MODEL_NAME)
    xlmr_model = AutoModelForSequenceClassification.from_pretrained(XLMR_MODEL_NAME)
    device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    xlmr_model.to(device).eval()

    infer_df   = expand_aspects(df)
    sentiments = []

    for i in range(0, len(infer_df), BATCH_SIZE):
        batch = infer_df["clean_text"].iloc[i : i + BATCH_SIZE].tolist()
        enc   = xlmr_tok(batch, padding=True, truncation=True,
                         max_length=MAX_LEN, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = xlmr_model(**enc).logits
        pred_ids = np.argmax(logits.cpu().numpy(), axis=1)
        sentiments.extend([
            XLMR_LABEL_MAP.get(xlmr_model.config.id2label[i], "neutral")
            for i in pred_ids
        ])

    infer_df["sentiment"] = sentiments
    return infer_df

# ─────────────────────────────────────────────
# 11. MERGE + SAVE submission.json
# ─────────────────────────────────────────────
print("\nRunning MARBERT on Arabic reviews...")
arabic_results = run_marbert(arabic_df)

print("Running XLM-R on non-Arabic reviews...")
xlmr_results   = run_xlmr(other_df)

all_results = pd.concat([arabic_results, xlmr_results], ignore_index=True)

submission = []
for review_id, group in all_results.groupby("review_id", sort=False):
    submission.append({
        "review_id":         int(review_id),
        "aspects":           group["aspect"].tolist(),
        "aspect_sentiments": dict(zip(group["aspect"], group["sentiment"])),
    })

submission.sort(key=lambda x: x["review_id"])

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print(f"\nDone — {len(submission)} reviews saved to {OUTPUT_PATH}")
print("Sample:", json.dumps(submission[0], ensure_ascii=False, indent=2))


Aspect classifier trained on 1971 reviews ✓
Train rows: 4043 | Val rows: 840
Label distribution (train):
 sentiment
positive    1816
negative    1774
neutral      453
Name: count, dtype: int64

Class weights ([np.str_('negative'), np.str_('neutral'), np.str_('positive')]): [0.7597, 2.975, 0.7421]


Map:   0%|          | 0/4043 [00:00<?, ? examples/s]

Map:   0%|          | 0/840 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

...

Map:   0%|          | 0/554 [00:00<?, ? examples/s]

Running XLM-R on non-Arabic reviews...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Done — 500 reviews saved to submission.json
Sample: {
  "review_id": 23,
  "aspects": [
    "service"
  ],
  "aspect_sentiments": {
    "service": "positive"
  }
}
